# 第7章：建立聊天應用程式
## Microsoft Foundry Models API 快速入門

本筆記本改編自 [Azure OpenAI 範例倉庫](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst)，該倉庫包含可存取 [Azure OpenAI](notebook-azure-openai.ipynb) 服務的筆記本。

> **注意：** GitHub Models 將於2026年7月底退休。本筆記本現以 [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) 為目標，提供相同的免費試用模型目錄及 Azure AI 推論 SDK 體驗。


# 概述  
「大型語言模型是將文字映射到文字的函式。給定一個輸入文字串，大型語言模型會嘗試預測接下來的文字」(1)。這個「快速入門」筆記本將向使用者介紹大型語言模型的高階概念、開始使用 AML 所需的核心套件、提示設計的基礎介紹，以及幾個不同使用案例的簡短範例。 


## 目錄  

[概覽](#overview)  
[如何使用 OpenAI 服務](#how-to-use-openai-service)  
[1. 建立您的 OpenAI 服務](#1.-creating-your-openai-service)  
[2. 安裝](#2.-installation)    
[3. 認證資料](#3.-credentials)  

[使用案例](#use-cases)    
[1. 摘要文本](#1.-summarize-text)  
[2. 分類文本](#2.-classify-text)  
[3. 生成新產品名稱](#3.-generate-new-product-names)  
[4. 微調分類器](#4.fine-tune-a-classifier)  

[參考資料](#references)


### 建立您的第一個提示語句  
這個簡短的練習將提供在 Microsoft Foundry Models 中提交提示語句給模型以完成簡單任務「摘要」的基本介紹。  


<strong>步驟</strong>：  
1. 若尚未安裝，請在您的 Python 環境中安裝 `azure-ai-inference` 函式庫。  
2. 載入標準輔助函式庫並設定您的 Microsoft Foundry Models 認證。  
3. 選擇適合任務的模型  
4. 為模型建立一個簡單的提示語句  
5. 向模型 API 提交您的請求！  


### 1. 安裝 `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. 匯入輔助函式庫並實例化認證


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. 找到合適的模型  
像 GPT-4o 和 GPT-4o mini 這樣的模型可以理解並生成自然語言，並且可在 Microsoft Foundry Models 目錄中找到，同時這個目錄還包含 Meta、Mistral、Cohere 和 Microsoft 的模型。 


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. 提示設計  

「大型語言模型的魔力在於通過在大量文本上訓練以最小化這種預測誤差，模型終究會學習到對這些預測有用的概念。例如，它們學會了類似」(1)：

* 如何拼寫
* 語法如何運作
* 如何改寫
* 如何回答問題
* 如何進行對話
* 如何用多種語言撰寫
* 如何編碼
* 等等。

#### 如何控制大型語言模型  
「在所有輸入到大型語言模型的資料中，文本提示(1) 是最具影響力的。

大型語言模型可以通過幾種方式被提示以產生輸出：

指令：告訴模型你需要什麼
補全：促使模型完成你想要的開頭部分
示範：向模型展示你需要的內容，可以是：
幾個在提示中的範例
幾百或幾千個用於微調訓練資料集的範例」



#### 建立提示的三項基本指南：

<strong>顯示並說明</strong>。透過指令、範例或兩者結合明確說明你想要什麼。如果你希望模型將一串項目按字母順序排序或根據情感分類段落，就要讓它明白這是你想要的。

<strong>提供優質資料</strong>。如果你想建立分類器或讓模型遵循某種模式，請確保有足夠的範例。務必校對你的範例——模型通常聰明到能識別基本拼寫錯誤並給出回應，但它也可能認為這是刻意的，而影響回應結果。

**檢查你的設定。** 溫度和 top_p 設定控制模型生成回應的決定性。如果你要求的回應只有一個正確答案，建議將這些值調低。若你想要更多樣化的回應，則可調高這些值。人們對這些設定最常犯的錯誤是誤以為它們是「聰明」或「創意」的控制開關。


來源：https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. 提交！


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### 重複相同的呼叫，結果有何不同？ 


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## 摘要文本  
#### 挑戰  
透過在文本段落末尾添加“tl;dr:”來摘要文本。注意模型如何在沒有額外指示的情況下理解並執行多項任務。你可以嘗試比 tl;dr 更具描述性的提示詞來調整模型的行為並自訂你所獲得的摘要(3)。  

最近的研究顯示，先在大型文本語料庫上進行預訓練，接著在特定任務上微調，可在許多自然語言處理任務和基準上大幅提升成果。雖然這類方法在架構上通常不依賴任務，但仍需數千至數萬個特定任務的微調數據集。相比之下，人類通常只需少數範例或簡單指令，就能執行新的語言任務——這是現有自然語言處理系統仍難以達成的。本研究展示，擴大語言模型規模能大幅提升任務無關的少量範例表現，有時甚至能媲美過去最先進的微調方法。  



摘要  


# 多種使用情境練習  
1. 文字摘要  
2. 文字分類  
3. 產生新產品名稱


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## 文字分類  
#### 挑戰  
將項目分類至推理時提供的類別。在以下範例中，我們在提示中同時提供類別和要分類的文字(*playground_reference)。 

客戶詢問：您好，我筆電鍵盤的一顆按鍵最近壞了，需要更換：

分類類別：


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## 產生新產品名稱
#### 挑戰
從範例詞彙創造產品名稱。這裡我們在提示中包含了我們要為其產生名稱的產品資訊。我們也提供了類似的範例，以展示我們希望收到的模式。我們還將溫度值設置得較高，以增加隨機性和更具創新性的回應。

產品描述：家用奶昔機
種子詞：快速、健康、緊湊。
產品名稱：HomeShaker、Fit Shaker、QuickShake、Shake Maker

產品描述：一雙可適配任何腳尺寸的鞋子。
種子詞：自適應、合腳、全適合。


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# 參考資料  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [微調 GPT 模型以分類文本的最佳實踐](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# 更多協助  
[OpenAI 商業化團隊](AzureOpenAITeam@microsoft.com) 


# 貢獻者
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
